## Experiment to identify memory capacity in the autoencoder RNN hidden state

### Recipe: experiment 1
- generate completely random sequence
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

### Recipe: experiment 2
- generate a sequence with pattern (1,2,3,4,5, 1,2,3,4,5, 1,2,3,4,5....)
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

In [21]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [22]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [23]:
def get_patterned_sequence(total_samples, token_number=7):
    sequence = []
    for i in range(total_samples):
        sequence.append(chr((i % token_number) + ord('A')))
    return sequence

In [24]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [25]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [26]:
# tokens start from 1
class Dataset_reconstructer(Dataset):
    def __init__(self, hidden_states, data, past_recall=1, short_term_memory=1):
        
        self.X = np.array(hidden_states)
        self.y = np.vectorize(ord)(data) - 64
        self.short_term_memory = short_term_memory
        self.past_recall = past_recall

        if short_term_memory == 1:
            self.y = np.concatenate(
                    (np.zeros(past_recall, dtype=int), self.y[:-past_recall])
                )

        self.X = tnsr(self.X)
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index+self.short_term_memory-self.past_recall-1]

    def __len__(self):
        return self.X.shape[0]

In [36]:
def path_finder_loss(y_next_hat, y_next_target, y_past_hat, y_past_target):
    ce1 = F.cross_entropy(y_next_hat.view(-1, y_next_hat.size(-1)), y_next_target.view(-1))
    ce2 = F.cross_entropy(y_past_hat.view(-1, y_past_hat.size(-1)), y_past_target.view(-1))

    return (ce1 + ce2)/2

In [37]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        self.linear_next = nn.Linear(hidden_size, vocab_size)
        self.linear_past = nn.Linear(hidden_size, vocab_size)

        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        next_pred = self.linear_next(out[:,-1,:])
        past_pred = self.linear_past(out[:,-1,:])

        return next_pred, past_pred, h  

In [38]:
class classifier(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, vocab_size)
        # self.linear2 = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = self.linear1(x)
        # out = self.linear2(x)

        return out  

In [57]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 10
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_random_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        h_prev = None
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, y_past, h = model(X)
            else:
                y_pred, y_past, h = model(X, h_prev)

            loss = criterion(y_pred[0], y[0], y_past, X[0][-2])
                 
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                h_prev = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, accuracy: {test_acc[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    _, _, h = model(X)
                else:
                    _, _, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_random_constrained.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, accuracy: 0.1600
Iter : 2001, accuracy: 0.1480
Iter : 3001, accuracy: 0.1400
Iter : 4001, accuracy: 0.1400
Iter : 5001, accuracy: 0.1340
Iter : 6001, accuracy: 0.1370
Iter : 7001, accuracy: 0.1370
Iter : 8001, accuracy: 0.1410
Iter : 9001, accuracy: 0.1400
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.12it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:09<00:00,  1.11it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9943983194958488
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 0.856241866052658
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.13it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.4980478526379017
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.12it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.28301131244368805


In [58]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 10
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_patterned_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        h_prev = None
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, y_past, h = model(X)
            else:
                y_pred, y_past, h = model(X, h_prev)

            loss = criterion(y_pred[0], y[0], y_past, X[0][-2])
                 
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                h_prev = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, accuracy: {test_acc[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    _, _, h = model(X)
                else:
                    _, _, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_patterned_constrained.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, accuracy: 0.9700
Iter : 2001, accuracy: 1.0000
Iter : 3001, accuracy: 1.0000
Iter : 4001, accuracy: 1.0000
Iter : 5001, accuracy: 1.0000
Iter : 6001, accuracy: 1.0000
Iter : 7001, accuracy: 1.0000
Iter : 8001, accuracy: 1.0000
Iter : 9001, accuracy: 1.0000
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.14it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.14it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9998999699909973
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.14it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.14it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0


In [59]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 10
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_sequence(total_samples, 2, 3, train_percent=1.0)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        h_prev = None
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, y_past, h = model(X)
            else:
                y_pred, y_past, h = model(X, h_prev)

            loss = criterion(y_pred[0], y[0], y_past, X[0][-2])
                 
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                h_prev = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, accuracy: {test_acc[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    _, _, h = model(X)
                else:
                    _, _, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_hard_patterned_constrained.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, accuracy: 0.4730
Iter : 2001, accuracy: 0.6490
Iter : 3001, accuracy: 0.6620
Iter : 4001, accuracy: 0.6640
Iter : 5001, accuracy: 0.6620
Iter : 6001, accuracy: 0.6740
Iter : 7001, accuracy: 0.6740
Iter : 8001, accuracy: 0.6720
Iter : 9001, accuracy: 0.6480
Doing recall  1


TypeError: ord() expected a character, but string of length 10000 found